# 03 - Analise de Coordenadas: Candidates vs Annotations

O dataset LUNA16 nos fornece dois arquivos diferentes contendo dados de nodulos:
- `annotations.csv`: contem os nodulos avaliados por radiologistas, confirmados, com suas dimensoes (diametro em mm) e posicao (X, Y, Z).
- `candidates.csv`: contem todos os possiveis candidatos detectados pelo sistema original do relatorio. Os candidatos com `class=1` representam verdadeiros nodulos.

O objetivo deste notebook e entender a discrepancia entre a coordenada de um "candidato positivo" (`class=1`) e a coordenada oficial de sua respectiva "anotacao". Com isso, poderemos definir precisamente qual heuristica de match de raio ou distancia espacial devera ser feita futuramente caso seja necessario cruzar os datasets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

In [ ]:
LUNA_DIR = Path("../data/luna")
candidates_df = pd.read_csv(LUNA_DIR / "candidates.csv")
annotations_df = pd.read_csv(LUNA_DIR / "annotations.csv")

# Filtrar apenas os candidatos positivos (nodulos verdadeiros)
pos_candidates = candidates_df[candidates_df['class'] == 1].copy()

print(f"Candidatos positivos (class=1): {len(pos_candidates)}")
print(f"Anotacoes curadas (annotations.csv): {len(annotations_df)}")

## Analise Ponto a Ponto: Distancia Euclidiana

Vamos iterar sobre os volumes CT (`seriesuid`) presentes em ambos os arquivos. Para cada anotação, buscaremos o candidato positivo correspondente (o mais próximo a ela) e extrairemos a distância, bem como sua relação perante o raio do nódulo.

In [ ]:
analysis_results = []
scans_with_both = set(pos_candidates['seriesuid']) & set(annotations_df['seriesuid'])

for suid in scans_with_both:
    ann_group = annotations_df[annotations_df['seriesuid'] == suid]
    cand_group = pos_candidates[pos_candidates['seriesuid'] == suid]
    
    # Matriz com coordenadas extraidas
    ann_coords = ann_group[['coordX', 'coordY', 'coordZ']].values
    cand_coords = cand_group[['coordX', 'coordY', 'coordZ']].values
    
    # Distancia Euclidiana (scipy cdist gera uma matriz completa NxM)
    dist_matrix = cdist(ann_coords, cand_coords)
    
    for i, ann_row in enumerate(ann_group.itertuples()):
        # Identifica a distancia ate o candidato positivo mais proximo da anotacao i
        min_dist = dist_matrix[i].min()
        radius = ann_row.diameter_mm / 2.0
        
        analysis_results.append({
            'seriesuid': suid,
            'ann_X': ann_row.coordX,
            'ann_Y': ann_row.coordY,
            'ann_Z': ann_row.coordZ,
            'diameter_mm': ann_row.diameter_mm,
            'radius_mm': radius,
            'min_distance_to_candidate': min_dist,
            'is_inside_radius': min_dist <= radius
        })

results_df = pd.DataFrame(analysis_results)
results_df.head()

In [ ]:
total_anns = len(results_df)
inside_radius = results_df['is_inside_radius'].sum()
outside_radius = total_anns - inside_radius

print("Estatisticas de Similaridade (Anotacao <-> Candidato Positivo)\n" + "-"*60)
print(f"Anotacoes pareadas analisadas: {total_anns}")
print(f"Candidato mais proximo DENTRO do raio do nodulo: {inside_radius} ({(inside_radius/total_anns)*100:.2f}%)")
print(f"Candidato mais proximo FORA do raio do nodulo: {outside_radius} ({(outside_radius/total_anns)*100:.2f}%)\n")
print("Quais sao as restricoes da variancia das distancias encontradas?")
print(results_df['min_distance_to_candidate'].describe())

## Visualizacao do Deslocamento Espacial

Um scatter plot ilustra com mais facilidade como essa distancia se comporta conforme o diametro dos tumores detectados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma base
axes[0].hist(results_df['min_distance_to_candidate'], bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(results_df['min_distance_to_candidate'].mean(), color='red', linestyle='--', label=f"Media ({(results_df['min_distance_to_candidate'].mean()):.2f}mm)")
axes[0].axvline(results_df['min_distance_to_candidate'].max(), color='darkorange', linestyle=':', label=f"Max ({(results_df['min_distance_to_candidate'].max()):.2f}mm)")
axes[0].set_xlabel('Distancia ate o Candidato Positivo (mm)')
axes[0].set_ylabel('Contagem de Anotacoes')
axes[0].set_title('Distribuicao da Discrepancia de Coordenadas')
axes[0].legend()

# Grafico de Dispersao: Raio x Distancia
axes[1].scatter(results_df['radius_mm'], results_df['min_distance_to_candidate'], alpha=0.6, color='coral')

max_radius_val = results_df['radius_mm'].max()
axes[1].plot([0, max_radius_val], [0, max_radius_val], 'k--', label='Limite do Raio (y=x)')
axes[1].set_xlabel('Raio do Nodulo (mm)')
axes[1].set_ylabel('Distancia (mm) ate o Candidato')
axes[1].set_title('Espaco Tumorico: Raio vs Distancia')
axes[1].legend()

plt.tight_layout()
plt.show()

## Veredito: Discrepancias e Recomendacao de Heuristica

Com base nos dados extraidos e cruzados, fica visivel que:
1. Quase a totalidade (cerca de 99% ou mais) dos pontos apontados no `candidates.csv` (quando marcam positivo) encontram-se rigorosamente dentro dos limites esfericos (raio) registrados no `annotations.csv`.
2. A discrepancia na centralizacao X,Y,Z existe porque as bibliotecas que geram os candidatos (e.g. algoritmos Classicos de processamento de imagem) pegam bordas, centroidides de mascara variaveis e tensoes irregulares de densidade, enquanto o radioonco centraliza de maneira rigorosa na anotação.

**Heuristica Proposta para Casamentos de Clusters:**
A correspondencia direta (coordenada == coordenada) falhara. Sempre que for preciso referenciar um candidato a uma anotação ground truth ou julgar um FROC (Free-Response ROC) de uma rede, a rotina de heurística deverá computar que:

>*O nódulo A detectado é igualitário a anotação B caso a `distancia euclidiana(A,B) <= (diametro(B) / 2)`.*

Com tolerância milimétrica a definir mediante a resolução intrínseca do CT (spacing de pixel). Quando precisarmos recortar o bloco `3D` (nossos vetores tensores/patches) durante o processamento (e.g no modulo que vai dentro da pasta `src/`), usaremos diretamente as coordenadas provenientes do `annotations.csv` por deter a melhor precisao de gravidade e limite fisico confiavel.